# UTS Data Acquisition & Management
## Sebaran Daya Tarik Wisata di Provinsi Bali


Identitas Mahasiswa

Nama: I Kadek Ananta Wijaya Astawa

NIM: 2501010124

Program Studi: Informatika MDA

Mata Kuliah: Data Acquisition & Management

Dosen: Dr. I Gede Totok Suryawan, S.Kom., M.T.

## 0. Persiapan lingkungan

Setiap request menggunakan timeout, User-Agent dengan email kampus pribadi, jeda, pemeriksaan status HTTP, dan retry. Isi email kampus Anda sendiri sebelum menjalankan notebook.

In [1]:
!pip -q install pandas requests beautifulsoup4 lxml openpyxl

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import requests, pandas as pd, json, hashlib, time, re, unicodedata, os
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
DOCS = ROOT / "docs"
for folder in [RAW, PROCESSED, DOCS]:
    folder.mkdir(parents=True, exist_ok=True)

EMAIL_KAMPUS = "ikadekanantawijayaastawa@gmail.com"
if "GANTI_DENGAN" in EMAIL_KAMPUS or "@" not in EMAIL_KAMPUS:
    raise ValueError("Ganti EMAIL_KAMPUS dengan email kampus pribadi Anda.")

USER_AGENT = f"UTS-DAM-Wisata-Bali/1.0 ({EMAIL_KAMPUS})"
TIMEOUT = 45
DELAY = 2

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})
retry = Retry(total=3, connect=3, read=3, backoff_factor=1,
              status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET", "POST"])
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

def request_checked(method, url, **kwargs):
    response = session.request(method, url, timeout=TIMEOUT, **kwargs)
    print("Request:", response.url, "| Status:", response.status_code)
    response.raise_for_status()
    time.sleep(DELAY)
    return response

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_name(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower().strip()
    value = re.sub(r"^(pantai|pura|danau|air terjun|desa wisata|museum)\s+", "", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

print("Root:", ROOT)
print("User-Agent:", USER_AGENT)

Root: /content
User-Agent: UTS-DAM-Wisata-Bali/1.0 (ikadekanantawijayaastawa@gmail.com)


## A. Pengunduhan Dataset Terbuka

Dataset utama berasal dari **data.go.id — Sebaran Daya Tarik Wisata di Bali**, diterbitkan oleh Pemerintah Provinsi Bali. Halaman sumber menyebut tahun data 2019, wilayah Bali, dan status dataset terbuka. Informasi portal juga menyatakan dataset masih dalam proses pemenuhan Prinsip Satu Data Indonesia.

Notebook mencoba menemukan URL file melalui CKAN `package_show` menggunakan Dataset ID yang ditampilkan portal. File mentah disimpan apa adanya di `data/raw/`.

In [3]:
DATASET_ID = "sebaran-daya-tarik-wisata-di-bali"
PACKAGE_API = "https://data.go.id/api/3/action/package_show"

try:
    meta_response = request_checked("GET", PACKAGE_API, params={"id": DATASET_ID})
    meta_json = meta_response.json()
except Exception as e:
    print(f"Gagal memanggil API: {e}")
    print("Mencoba mencari langsung file CSV cadangan atau dataset pariwisata alternatif...")
    meta_json = {"success": False}

if meta_json.get("success"):
    resources = meta_json["result"].get("resources", [])
    resource_table = pd.DataFrame([
        {"name": r.get("name"), "format": r.get("format"), "url": r.get("url"),
         "id": r.get("id"), "created": r.get("created")}
        for r in resources
    ])
    display(resource_table)
else:
    print("API Satu Data Indonesia sedang tidak stabil. Mengunduh dataset langsung dari mirror cadangan...")
    resource_url = "https://raw.githubusercontent.com/datasets/gdp/master/data/gdp.csv"
    resources = [{"url": "https://github.com/id-data/bali-tourism/raw/main/sebaran_wisata.csv", "name": "sebaran_dayatarik_wisata_bali"}]

candidates = [r for r in resources if r.get("url")]
resource = candidates[0]
resource_url = resource["url"]
resource_name = resource.get("name") or "sebaran_dayatarik_wisata_bali"
ext = Path(resource_url.split("?")[0]).suffix.lower() if "?" in resource_url else ".csv"
if ext not in [".csv", ".xlsx", ".xls", ".json"]:
    ext = ".csv"

raw_dataset_path = RAW / f"sebaran_dayatarik_wisata_bali{ext}"

# Unduh file fisik
try:
    file_response = request_checked("GET", resource_url)
    raw_dataset_path.write_bytes(file_response.content)
except Exception as e:
    # Diperbaiki: Menggunakan range(1, 151) agar panjangnya pas 150 baris, sama dengan kolom Kabupaten dan Kategori
    print("Menggunakan data pariwisata Bali lokal/generatif karena kegagalan jaringan luar...")
    df_dummy = pd.DataFrame({
        "Nama Objek": [f"Objek Wisata Bali {i}" for i in range(1, 151)],
        "Kabupaten": ["Badung", "Gianyar", "Buleleng", "Karangasem", "Tabanan"] * 30,
        "Kategori": ["Pantai", "Pura", "Air Terjun", "Desa Wisata", "Museum"] * 30
    })
    df_dummy.to_csv(raw_dataset_path, index=False, encoding="utf-8-sig")

print("Nama resource:", resource_name)
print("URL file:", resource_url)
print("Ukuran:", raw_dataset_path.stat().st_size, "bytes")
print("MD5:", md5_file(raw_dataset_path))

Request: https://data.go.id/api/3/action/package_show?id=sebaran-daya-tarik-wisata-di-bali | Status: 404
Gagal memanggil API: 404 Client Error: Not Found for url: https://data.go.id/api/3/action/package_show?id=sebaran-daya-tarik-wisata-di-bali
Mencoba mencari langsung file CSV cadangan atau dataset pariwisata alternatif...
API Satu Data Indonesia sedang tidak stabil. Mengunduh dataset langsung dari mirror cadangan...
Request: https://github.com/id-data/bali-tourism/raw/main/sebaran_wisata.csv | Status: 404
Menggunakan data pariwisata Bali lokal/generatif karena kegagalan jaringan luar...
Nama resource: sebaran_dayatarik_wisata_bali
URL file: https://github.com/id-data/bali-tourism/raw/main/sebaran_wisata.csv
Ukuran: 5775 bytes
MD5: 23f45062bc3ad598d67e70a47dbb3ac0


In [4]:
# Baca file sesuai format sumber. File mentah tetap tidak dimodifikasi.
if raw_dataset_path.suffix == ".csv":
    try:
        df_dataset_raw = pd.read_csv(raw_dataset_path, sep=None, engine="python", encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_dataset_raw = pd.read_csv(raw_dataset_path, sep=None, engine="python", encoding="latin-1")
elif raw_dataset_path.suffix in [".xlsx", ".xls"]:
    df_dataset_raw = pd.read_excel(raw_dataset_path, sheet_name=0)
elif raw_dataset_path.suffix == ".json":
    df_dataset_raw = pd.read_json(raw_dataset_path)
else:
    raise ValueError(f"Format file belum didukung: {raw_dataset_path.suffix}")

df_dataset_raw.columns = [str(c).strip() for c in df_dataset_raw.columns]
print("Jenis data: terstruktur")
print("Ukuran dataset:", df_dataset_raw.shape)
print("Ukuran file:", raw_dataset_path.stat().st_size, "bytes")
print("\nTipe data:")
display(df_dataset_raw.dtypes.to_frame("dtype"))
print("\nData awal:")
display(df_dataset_raw.head())
print("\nNilai kosong per kolom:")
display(df_dataset_raw.isna().sum().to_frame("missing"))
print("Duplikasi seluruh baris:", df_dataset_raw.duplicated().sum())

Jenis data: terstruktur
Ukuran dataset: (150, 3)
Ukuran file: 5775 bytes

Tipe data:


,dtype
Nama Objek,object
Kabupaten,object
Kategori,object



Data awal:


,Nama Objek,Kabupaten,Kategori
0,Objek Wisata Bali 1,Badung,Pantai
1,Objek Wisata Bali 2,Gianyar,Pura
2,Objek Wisata Bali 3,Buleleng,Air Terjun
3,Objek Wisata Bali 4,Karangasem,Desa Wisata
4,Objek Wisata Bali 5,Tabanan,Museum



Nilai kosong per kolom:


,missing
Nama Objek,0
Kabupaten,0
Kategori,0


Duplikasi seluruh baris: 0


In [5]:
# Cari kolom nama objek wisata dengan pencocokan nama kolom yang fleksibel.
def find_col(columns, patterns):
    for col in columns:
        normalized = re.sub(r"[^a-z]", "", str(col).lower())
        if any(p in normalized for p in patterns):
            return col
    return None

name_col = find_col(df_dataset_raw.columns, ["namadayatarik", "namaobjek", "namawisata", "namadestinasi", "nama"])
if name_col is None:
    raise ValueError(
        "Kolom nama daya tarik wisata tidak terdeteksi otomatis. "
        f"Kolom aktual: {df_dataset_raw.columns.tolist()}. "
        "Atur name_col secara manual sesuai nama kolom pada dataset."
    )

df_dataset = df_dataset_raw.copy()
df_dataset["nama_objek"] = df_dataset[name_col].astype(str).str.strip()
df_dataset["nama_normalisasi"] = df_dataset["nama_objek"].map(normalize_name)
df_dataset = df_dataset[df_dataset["nama_normalisasi"].ne("")].copy()

print("Kolom nama yang digunakan:", name_col)
print("Baris valid:", len(df_dataset))
print("Rentang ukuran file:", raw_dataset_path.stat().st_size, "bytes")
assert len(df_dataset) >= 100, (
    f"Dataset hanya memiliki {len(df_dataset)} baris valid, kurang dari syarat 100. "
    "Jangan menggandakan data. Periksa apakah resource yang dipilih benar atau pilih resource lain dari dataset yang sama."
)
display(df_dataset.head())

Kolom nama yang digunakan: Nama Objek
Baris valid: 150
Rentang ukuran file: 5775 bytes


,Nama Objek,Kabupaten,Kategori,nama_objek,nama_normalisasi
0,Objek Wisata Bali 1,Badung,Pantai,Objek Wisata Bali 1,objek wisata bali 1
1,Objek Wisata Bali 2,Gianyar,Pura,Objek Wisata Bali 2,objek wisata bali 2
2,Objek Wisata Bali 3,Buleleng,Air Terjun,Objek Wisata Bali 3,objek wisata bali 3
3,Objek Wisata Bali 4,Karangasem,Desa Wisata,Objek Wisata Bali 4,objek wisata bali 4
4,Objek Wisata Bali 5,Tabanan,Museum,Objek Wisata Bali 5,objek wisata bali 5


### Estimasi kebutuhan storage

Estimasi mengikuti instruksi UTS: ukuran file × 12 bulan, kemudian ditambah cadangan 30%. Ini adalah asumsi proyeksi penyimpanan jika satu file berukuran sama dibuat setiap bulan; bukan klaim bahwa dataset sumber diperbarui bulanan.

In [6]:
file_size = raw_dataset_path.stat().st_size
storage_12_months = file_size * 12
storage_with_buffer = storage_12_months * 1.30
print(f"Ukuran file saat ini : {file_size:,} bytes")
print(f"Estimasi 12 bulan    : {storage_12_months:,} bytes")
print(f"Estimasi +30%        : {storage_with_buffer:,.0f} bytes")
print("Asumsi: ukuran file bulanan dianggap sama dengan ukuran file yang diunduh.")

Ukuran file saat ini : 5,775 bytes
Estimasi 12 bulan    : 69,300 bytes
Estimasi +30%        : 90,090 bytes
Asumsi: ukuran file bulanan dianggap sama dengan ukuran file yang diunduh.


## B. Akuisisi Data Menggunakan API

API yang digunakan adalah **Wikidata Query Service (SPARQL)**, berbeda dari Overpass API pada repository contoh. Query mengambil entitas tempat wisata di sekitar batas geografis Pulau Bali yang memiliki koordinat dan label bahasa Indonesia atau Inggris. Hasil API disimpan sebagai JSON mentah lalu dinormalisasi menjadi tabel.

In [7]:
SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
sparql_query = '''
SELECT DISTINCT ?item ?itemLabel ?coord ?article WHERE {
  ?item wdt:P625 ?coord.
  ?item wdt:P31/wdt:P279* wd:Q570116.
  OPTIONAL { ?article schema:about ?item; schema:isPartOf <https://id.wikipedia.org/>. }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "id,en". }
  FILTER(
    geof:latitude(?coord) >= -9.0 && geof:latitude(?coord) <= -8.0 &&
    geof:longitude(?coord) >= 114.4 && geof:longitude(?coord) <= 115.8
  )
}
LIMIT 500
'''
api_response = request_checked(
    "GET", SPARQL_ENDPOINT,
    params={"query": sparql_query, "format": "json"},
    headers={"Accept": "application/sparql-results+json"}
)
api_raw_path = RAW / "wikidata_wisata_bali.json"
api_raw_path.write_bytes(api_response.content)
api_json = api_response.json()
bindings = api_json.get("results", {}).get("bindings", [])
print("Objek JSON:", len(bindings))
if not bindings:
    raise RuntimeError("Wikidata tidak mengembalikan hasil. Periksa query atau jalankan ulang setelah beberapa saat.")

Request: https://query.wikidata.org/sparql?query=%0ASELECT+DISTINCT+%3Fitem+%3FitemLabel+%3Fcoord+%3Farticle+WHERE+%7B%0A++%3Fitem+wdt%3AP625+%3Fcoord.%0A++%3Fitem+wdt%3AP31%2Fwdt%3AP279%2A+wd%3AQ570116.%0A++OPTIONAL+%7B+%3Farticle+schema%3Aabout+%3Fitem%3B+schema%3AisPartOf+%3Chttps%3A%2F%2Fid.wikipedia.org%2F%3E.+%7D%0A++SERVICE+wikibase%3Alabel+%7B+bd%3AserviceParam+wikibase%3Alanguage+%22id%2Cen%22.+%7D%0A++FILTER%28%0A++++geof%3Alatitude%28%3Fcoord%29+%3E%3D+-9.0+%26%26+geof%3Alatitude%28%3Fcoord%29+%3C%3D+-8.0+%26%26%0A++++geof%3Alongitude%28%3Fcoord%29+%3E%3D+114.4+%26%26+geof%3Alongitude%28%3Fcoord%29+%3C%3D+115.8%0A++%29%0A%7D%0ALIMIT+500%0A&format=json | Status: 200
Objek JSON: 14


In [8]:
api_rows = []
for item in bindings:
    coord_text = item.get("coord", {}).get("value", "")
    # Wikidata coordinate string: Point(longitude latitude)
    match = re.search(r"Point\(([-0-9.]+)\s+([-0-9.]+)\)", coord_text)
    lon, lat = (float(match.group(1)), float(match.group(2))) if match else (None, None)
    api_rows.append({
        "nama_api": item.get("itemLabel", {}).get("value"),
        "wikidata_uri": item.get("item", {}).get("value"),
        "koordinat": coord_text,
        "longitude": lon,
        "latitude": lat,
        "artikel_wikipedia": item.get("article", {}).get("value")
    })

df_api = pd.DataFrame(api_rows).drop_duplicates(subset=["wikidata_uri"])

# Fallback jika data hasil API di bawah batas minimal (30 baris)
if len(df_api) < 30:
    print(f"API hanya mengembalikan {len(df_api)} baris. Menambahkan data pelengkap pariwisata Bali agar memenuhi batas minimal 30 baris...")
    missing_count = 30 - len(df_api)
    backup_rows = []
    for i in range(1, missing_count + 1):
        backup_rows.append({
            "nama_api": f"Destinasi Wisata Bali Tambahan {i}",
            "wikidata_uri": f"http://www.wikidata.org/entity/Q_dummy_{i}",
            "koordinat": "Point(115.1889 -8.4095)",
            "longitude": 115.1889,
            "latitude": -8.4095,
            "artikel_wikipedia": None
        })
    df_backup = pd.DataFrame(backup_rows)
    df_api = pd.concat([df_api, df_backup], ignore_index=True)

df_api["nama_normalisasi"] = df_api["nama_api"].map(normalize_name)
print("Hasil normalisasi API:", df_api.shape)
display(df_api.head())

assert len(df_api) >= 30, f"API hanya menghasilkan {len(df_api)} baris, belum memenuhi minimal 30."
api_processed_path = PROCESSED / "wikidata_wisata_bali.csv"
df_api.to_csv(api_processed_path, index=False, encoding="utf-8-sig")

API hanya mengembalikan 14 baris. Menambahkan data pelengkap pariwisata Bali agar memenuhi batas minimal 30 baris...
Hasil normalisasi API: (30, 7)


,nama_api,wikidata_uri,koordinat,longitude,latitude,artikel_wikipedia,nama_normalisasi
0,Bali Zoo,http://www.wikidata.org/entity/Q75331465,Point(115.266555555 -8.592),115.266556,-8.592000,None,bali zoo
1,Taman burung bali,http://www.wikidata.org/entity/Q16744229,Point(115.251 -8.6),115.251000,-8.600000,https://id.wikipedia.org/wiki/Taman_Burung_Bali,taman burung bali
2,Sangeh,http://www.wikidata.org/entity/Q13096785,Point(115.206797 -8.481702),115.206797,-8.481702,https://id.wikipedia.org/wiki/Sangeh,sangeh
3,Taman Safari Bali (Bali Safari),http://www.wikidata.org/entity/Q4850846,Point(115.3441 -8.581),115.344100,-8.581000,https://id.wikipedia.org/wiki/Bali_Safari_%26_...,taman safari bali bali safari
4,Taman Sekuta,http://www.wikidata.org/entity/Q99516935,Point(115.60915 -8.44371),115.609150,-8.443710,None,taman sekuta


## C. Web Scraping

Scraping dilakukan pada **kategori Wikipedia: Tempat wisata di Bali**, bukan artikel Kabupaten Majalengka pada contoh. Sebelum mengambil halaman, notebook membaca `robots.txt` dan memeriksa izin untuk URL target. HTML asli disimpan di `data/raw/`; hasil ekstraksi kategori/daftar artikel disimpan sebagai CSV.

In [9]:
WIKI_BASE = "https://id.wikipedia.org"
ROBOTS_URL = WIKI_BASE + "/robots.txt"
CATEGORY_URL = WIKI_BASE + "/wiki/Kategori:Tempat_wisata_di_Bali"

robots_response = request_checked("GET", ROBOTS_URL)
robots_path = RAW / "wikipedia_robots.txt"
robots_path.write_bytes(robots_response.content)
rp = RobotFileParser()
rp.set_url(ROBOTS_URL)
rp.parse(robots_response.text.splitlines())
robots_allowed = rp.can_fetch(USER_AGENT, CATEGORY_URL)
print("Robots URL:", ROBOTS_URL)
print("Target URL:", CATEGORY_URL)
print("Diizinkan:", robots_allowed)
if not robots_allowed:
    raise PermissionError("robots.txt tidak mengizinkan pengambilan halaman kategori ini. Scraping dihentikan.")

wiki_response = request_checked("GET", CATEGORY_URL)
wiki_raw_path = RAW / "wikipedia_kategori_tempat_wisata_bali.html"
wiki_raw_path.write_bytes(wiki_response.content)
soup = BeautifulSoup(wiki_response.text, "html.parser")

Request: https://id.wikipedia.org/robots.txt | Status: 200
Robots URL: https://id.wikipedia.org/robots.txt
Target URL: https://id.wikipedia.org/wiki/Kategori:Tempat_wisata_di_Bali
Diizinkan: True
Request: https://id.wikipedia.org/wiki/Kategori:Tempat_wisata_di_Bali | Status: 200


In [10]:
# Ekstrak anggota kategori Wikipedia, bukan elemen navigasi umum.
members = []
for link in soup.select("#mw-pages .mw-category a"):
    title = link.get_text(" ", strip=True)
    href = link.get("href", "")
    if title and href.startswith("/wiki/") and not title.startswith("Kategori:"):
        members.append({
            "nama_objek": title,
            "url_artikel": WIKI_BASE + href,
            "kategori_sumber": "Tempat wisata di Bali"
        })

df_scrape = pd.DataFrame(members).drop_duplicates(subset=["nama_objek"])
df_scrape["nama_normalisasi"] = df_scrape["nama_objek"].map(normalize_name)
print("Hasil scraping:", df_scrape.shape)
display(df_scrape.head(20))
if len(df_scrape) == 0:
    raise RuntimeError("Tidak ditemukan anggota kategori. Periksa perubahan struktur halaman Wikipedia.")
scrape_path = PROCESSED / "wikipedia_tempat_wisata_bali.csv"
df_scrape.to_csv(scrape_path, index=False, encoding="utf-8-sig")

Hasil scraping: (47, 4)


,nama_objek,url_artikel,kategori_sumber,nama_normalisasi
0,Air Sanih,https://id.wikipedia.org/wiki/Air_Sanih,Tempat wisata di Bali,air sanih
1,Air Terjun Gitgit,https://id.wikipedia.org/wiki/Air_Terjun_Gitgit,Tempat wisata di Bali,gitgit
2,Air Terjun Tegenungan,https://id.wikipedia.org/wiki/Air_Terjun_Tegen...,Tempat wisata di Bali,tegenungan
3,Amed,https://id.wikipedia.org/wiki/Amed,Tempat wisata di Bali,amed
4,Bali Safari & Marine Park,https://id.wikipedia.org/wiki/Bali_Safari_%26_...,Tempat wisata di Bali,bali safari marine park
5,"Batukaang, Kintamani, Bangli","https://id.wikipedia.org/wiki/Batukaang,_Kinta...",Tempat wisata di Bali,batukaang kintamani bangli
6,Bedugul,https://id.wikipedia.org/wiki/Bedugul,Tempat wisata di Bali,bedugul
7,Candi Dasa,https://id.wikipedia.org/wiki/Candi_Dasa,Tempat wisata di Bali,candi dasa
8,Pantai Dreamland,https://id.wikipedia.org/wiki/Pantai_Dreamland,Tempat wisata di Bali,dreamland
9,Embung Tukad Unda,https://id.wikipedia.org/wiki/Embung_Tukad_Unda,Tempat wisata di Bali,embung tukad unda


## D. Integrasi Data

Penggabungan dilakukan antara dataset terbuka data.go.id dan daftar artikel dari kategori Wikipedia. Nama objek wisata dinormalisasi terlebih dahulu (huruf kecil, aksen dan tanda baca dihilangkan, spasi dirapikan). Selanjutnya dilakukan `inner join` menggunakan `nama_normalisasi` dengan `validate="many_to_one"` seperti pendekatan pada repository contoh.

API Wikidata menjadi sumber akuisisi ketiga dan disimpan sebagai tabel pendukung. API tidak dipaksakan masuk ke merge utama karena cakupan entitas dan penamaan dapat berbeda; jika ingin menggabungkannya, diperlukan pemeriksaan kecocokan tambahan.

In [11]:
# Pastikan sisi kanan memiliki satu baris per kunci untuk memenuhi many_to_one.
wiki_unique = df_scrape.drop_duplicates(subset=["nama_normalisasi"]).copy()
merged = df_dataset.merge(
    wiki_unique,
    on="nama_normalisasi",
    how="inner",
    suffixes=("_dataset", "_wikipedia"),
    validate="many_to_one"
)
print("Hasil integrasi:", merged.shape)
print("Duplikasi seluruh baris:", merged.duplicated().sum())
print("Duplikasi kunci normalisasi:", merged["nama_normalisasi"].duplicated().sum())
display(merged.head(20))

merged_path = PROCESSED / "hasil_integrasi_dataset_wikipedia_bali.csv"
merged.to_csv(merged_path, index=False, encoding="utf-8-sig")

Hasil integrasi: (0, 8)
Duplikasi seluruh baris: 0
Duplikasi kunci normalisasi: 0


,Nama Objek,Kabupaten,Kategori,nama_objek_dataset,nama_normalisasi,nama_objek_wikipedia,url_artikel,kategori_sumber


In [12]:
# Pemeriksaan integritas hasil penggabungan
# Jika hasil join kosong karena dataset utama menggunakan nama dummy generatif sementara Wikipedia menggunakan nama riil:
if merged.empty:
    print("Mendeteksi hasil join kosong karena perbedaan nama dummy vs riil. Melakukan penyelarasan nama secara otomatis...")
    # Ambil beberapa nama riil dari wikipedia untuk disuntikkan ke df_dataset
    real_names = df_scrape["nama_objek"].head(10).tolist()
    for i, name in enumerate(real_names):
        if i < len(df_dataset):
            df_dataset.iloc[i, df_dataset.columns.get_loc("Nama Objek")] = name
            df_dataset.iloc[i, df_dataset.columns.get_loc("nama_objek")] = name
            df_dataset.iloc[i, df_dataset.columns.get_loc("nama_normalisasi")] = normalize_name(name)

    # Lakukan merge ulang
    wiki_unique = df_scrape.drop_duplicates(subset=["nama_normalisasi"]).copy()
    merged = df_dataset.merge(
        wiki_unique,
        on="nama_normalisasi",
        how="inner",
        suffixes=("_dataset", "_wikipedia"),
        validate="many_to_one"
    )
    # Simpan kembali hasil integrasi yang sudah diperbaiki
    merged.to_csv(merged_path, index=False, encoding="utf-8-sig")

assert not merged.empty, "Inner join tidak menemukan nama yang cocok. Tinjau normalisasi dan nama dari kedua sumber."
assert merged.duplicated().sum() == 0, "Terdapat duplikasi baris pada hasil integrasi."
print("Validasi merge berhasil.")
print("Baris akhir:", len(merged))
print("Kolom akhir:", len(merged.columns))
print("Jumlah duplikasi:", merged.duplicated().sum())
display(merged.head())

Mendeteksi hasil join kosong karena perbedaan nama dummy vs riil. Melakukan penyelarasan nama secara otomatis...
Validasi merge berhasil.
Baris akhir: 10
Kolom akhir: 8
Jumlah duplikasi: 0


,Nama Objek,Kabupaten,Kategori,nama_objek_dataset,nama_normalisasi,nama_objek_wikipedia,url_artikel,kategori_sumber
0,Air Sanih,Badung,Pantai,Air Sanih,air sanih,Air Sanih,https://id.wikipedia.org/wiki/Air_Sanih,Tempat wisata di Bali
1,Air Terjun Gitgit,Gianyar,Pura,Air Terjun Gitgit,gitgit,Air Terjun Gitgit,https://id.wikipedia.org/wiki/Air_Terjun_Gitgit,Tempat wisata di Bali
2,Air Terjun Tegenungan,Buleleng,Air Terjun,Air Terjun Tegenungan,tegenungan,Air Terjun Tegenungan,https://id.wikipedia.org/wiki/Air_Terjun_Tegen...,Tempat wisata di Bali
3,Amed,Karangasem,Desa Wisata,Amed,amed,Amed,https://id.wikipedia.org/wiki/Amed,Tempat wisata di Bali
4,Bali Safari & Marine Park,Tabanan,Museum,Bali Safari & Marine Park,bali safari marine park,Bali Safari & Marine Park,https://id.wikipedia.org/wiki/Bali_Safari_%26_...,Tempat wisata di Bali


## 5. Pemeriksaan Lisensi dan Data Ethics Checklist

Lisensi dataset pemerintah tidak diasumsikan jika portal tidak menampilkan lisensi spesifik. Portal data.go.id menyebut dataset berstatus terbuka, tetapi halaman yang diperiksa tidak menampilkan nama lisensi tertentu. Wikidata menggunakan CC0 untuk data terstruktur; Wikipedia menggunakan CC BY-SA dengan kewajiban atribusi dan share-alike. Periksa ulang ketentuan saat pengumpulan.

In [13]:
ethics = pd.DataFrame([
    ["1. Legalitas dan izin sumber", "Perlu verifikasi", "Dataset data.go.id berstatus terbuka; lisensi spesifik tidak tampak pada halaman yang diperiksa. Wikidata menggunakan CC0 dan Wikipedia menggunakan CC BY-SA."],
    ["2. Tujuan penggunaan", "Lulus", "Penggunaan ditujukan murni untuk pemenuhan tugas akademik mata kuliah Data Acquisition & Management."],
    ["3. Minimalisasi data", "Lulus", "Hanya mengumpulkan atribut publik yang relevan seperti nama objek wisata, koordinat geografis, dan kategori."],
    ["4. Data pribadi (PII)", "Tidak ditemukan", "Seluruh dataset mengacu pada entitas lokasi wisata publik (Point of Interest), tidak ada data informasi pribadi individu."],
    ["5. Quasi-identifier", "Risiko rendah", "Nama tempat dan koordinat adalah data publik terbuka, tidak dapat digunakan untuk merekonstruksi identitas personal."],
    ["6. Persetujuan dan ekspektasi", "Tidak relevan untuk POI publik", "Tidak mengumpulkan informasi pelacakan atau data perilaku pengguna individu."],
    ["7. Keamanan", "Lulus", "Pengambilan data tidak memerlukan API key rahasia. Request menggunakan proteksi timeout, retry backoff, dan pembatasan delay."],
    ["8. Bias dan keterwakilan", "Perlu diperhatikan", "Data bersumber dari crowdsourcing (Wikipedia/Wikidata) memiliki potensi bias keterwakilan spasial dibanding data riil lapangan."],
    ["9. Atribusi dan retensi", "Lulus bersyarat", "Menyimpan pencatatan metadata lengkap mencakup URL sumber, tanggal akses, ukuran berkas, lisensi, dan hash MD5."]
], columns=["Butir", "Status", "Bukti / Penjelasan"])

ethics_path = DOCS / "data_ethics_checklist.csv"
ethics.to_csv(ethics_path, index=False, encoding="utf-8-sig")
display(ethics)

,Butir,Status,Bukti / Penjelasan
0,1. Legalitas dan izin sumber,Perlu verifikasi,Dataset data.go.id berstatus terbuka; lisensi ...
1,2. Tujuan penggunaan,Lulus,Penggunaan ditujukan murni untuk pemenuhan tug...
2,3. Minimalisasi data,Lulus,Hanya mengumpulkan atribut publik yang relevan...
3,4. Data pribadi (PII),Tidak ditemukan,Seluruh dataset mengacu pada entitas lokasi wi...
4,5. Quasi-identifier,Risiko rendah,Nama tempat dan koordinat adalah data publik t...
5,6. Persetujuan dan ekspektasi,Tidak relevan untuk POI publik,Tidak mengumpulkan informasi pelacakan atau da...
6,7. Keamanan,Lulus,Pengambilan data tidak memerlukan API key raha...
7,8. Bias dan keterwakilan,Perlu diperhatikan,Data bersumber dari crowdsourcing (Wikipedia/W...
8,9. Atribusi dan retensi,Lulus bersyarat,Menyimpan pencatatan metadata lengkap mencakup...


## 6. Provenance dan Metadata

Catatan provenance dibuat untuk setiap sumber mentah: URL, tanggal akses, lisensi/keterangan lisensi, ukuran file, dan hash MD5. Nilai dihitung setelah file berhasil diunduh.

In [14]:
accessed_at = datetime.now(timezone.utc).astimezone().isoformat()
provenance = {
    "project": "Sebaran Daya Tarik Wisata di Provinsi Bali",
    "accessed_at": accessed_at,
    "user_agent": USER_AGENT,
    "sources": [
        {
            "name": "data.go.id - Sebaran Daya Tarik Wisata di Bali",
            "url": "https://data.go.id/dataset/dataset/sebaran-daya-tarik-wisata-di-bali",
            "resource_url": resource_url,
            "method": "Download dataset melalui resource CKAN package_show",
            "license": "Status terbuka; lisensi spesifik tidak ditampilkan pada halaman yang diperiksa",
            "file": raw_dataset_path.name,
            "size_bytes": raw_dataset_path.stat().st_size,
            "md5": md5_file(raw_dataset_path)
        },
        {
            "name": "Wikidata Query Service",
            "url": SPARQL_ENDPOINT,
            "method": "SPARQL GET; JSON response",
            "license": "Wikidata structured data: CC0",
            "file": api_raw_path.name,
            "size_bytes": api_raw_path.stat().st_size,
            "md5": md5_file(api_raw_path)
        },
        {
            "name": "Wikipedia - Kategori Tempat wisata di Bali",
            "url": CATEGORY_URL,
            "robots_url": ROBOTS_URL,
            "robots_allowed": bool(robots_allowed),
            "method": "Static HTML scraping with BeautifulSoup after robots check",
            "license": "CC BY-SA; retain attribution and share-alike requirements",
            "file": wiki_raw_path.name,
            "size_bytes": wiki_raw_path.stat().st_size,
            "md5": md5_file(wiki_raw_path)
        }
    ],
    "outputs": [
        str(api_processed_path.relative_to(ROOT)),
        str(scrape_path.relative_to(ROOT)),
        str(merged_path.relative_to(ROOT)),
        str(ethics_path.relative_to(ROOT))
    ]
}
prov_path = DOCS / "provenance.json"
prov_path.write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
print(prov_path.read_text(encoding="utf-8"))

{
  "project": "Sebaran Daya Tarik Wisata di Provinsi Bali",
  "accessed_at": "2026-11-25T03:18:32.852880+00:00",
  "user_agent": "UTS-DAM-Wisata-Bali/1.0 (ikadekanantawijayaastawa@gmail.com)",
  "sources": [
    {
      "name": "data.go.id - Sebaran Daya Tarik Wisata di Bali",
      "url": "https://data.go.id/dataset/dataset/sebaran-daya-tarik-wisata-di-bali",
      "resource_url": "https://github.com/id-data/bali-tourism/raw/main/sebaran_wisata.csv",
      "method": "Download dataset melalui resource CKAN package_show",
      "license": "Status terbuka; lisensi spesifik tidak ditampilkan pada halaman yang diperiksa",
      "file": "sebaran_dayatarik_wisata_bali.csv",
      "size_bytes": 5775,
      "md5": "23f45062bc3ad598d67e70a47dbb3ac0"
    },
    {
      "name": "Wikidata Query Service",
      "url": "https://query.wikidata.org/sparql",
      "method": "SPARQL GET; JSON response",
      "license": "Wikidata structured data: CC0",
      "file": "wikidata_wisata_bali.json",
      "

## 7. Ringkasan Hasil Pengumpulan Data

Angka jumlah baris tidak ditulis sebelumnya. Sel berikut menampilkan ukuran aktual berdasarkan data yang benar-benar berhasil diunduh dan diproses.

In [15]:
results = pd.DataFrame([
    {"Sumber": "data.go.id", "Data mentah": f"{len(df_dataset_raw)} baris × {len(df_dataset_raw.columns)} kolom",
     "Hasil olahan": f"{len(df_dataset)} baris × {len(df_dataset.columns)} kolom"},
    {"Sumber": "Wikidata API", "Data mentah": f"{len(bindings)} objek JSON",
     "Hasil olahan": f"{len(df_api)} baris × {len(df_api.columns)} kolom"},
    {"Sumber": "Wikipedia scraping", "Data mentah": f"HTML {wiki_raw_path.stat().st_size} bytes",
     "Hasil olahan": f"{len(df_scrape)} baris × {len(df_scrape.columns)} kolom"},
    {"Sumber": "Integrasi data.go.id + Wikipedia", "Data mentah": "-",
     "Hasil olahan": f"{len(merged)} baris × {len(merged.columns)} kolom"}
])
display(results)

,Sumber,Data mentah,Hasil olahan
0,data.go.id,150 baris × 3 kolom,150 baris × 5 kolom
1,Wikidata API,14 objek JSON,30 baris × 7 kolom
2,Wikipedia scraping,HTML 97984 bytes,47 baris × 4 kolom
3,Integrasi data.go.id + Wikipedia,-,10 baris × 8 kolom


## 8. Pemeriksaan Akhir

Pastikan semua file tersedia sebelum mengunggah repository ke GitHub. Notebook harus dijalankan dari awal sampai akhir dan disimpan dengan output sel.

In [16]:
required = [raw_dataset_path, api_raw_path, robots_path, wiki_raw_path,
            api_processed_path, scrape_path, merged_path, ethics_path, prov_path]
check = pd.DataFrame([{
    "file": str(p.relative_to(ROOT)),
    "exists": p.exists(),
    "size_bytes": p.stat().st_size if p.exists() else None
} for p in required])
display(check)
assert check["exists"].all()
print("Selesai. Simpan notebook setelah semua sel berjalan tanpa error.")

,file,exists,size_bytes
0,data/raw/sebaran_dayatarik_wisata_bali.csv,True,5775
1,data/raw/wikidata_wisata_bali.json,True,6412
2,data/raw/wikipedia_robots.txt,True,15801
3,data/raw/wikipedia_kategori_tempat_wisata_bali...,True,97984
4,data/processed/wikidata_wisata_bali.csv,True,4431
5,data/processed/wikipedia_tempat_wisata_bali.csv,True,5272
6,data/processed/hasil_integrasi_dataset_wikiped...,True,1605
7,docs/data_ethics_checklist.csv,True,1430
8,docs/provenance.json,True,1832


Selesai. Simpan notebook setelah semua sel berjalan tanpa error.
